In [0]:
# Databricks notebook source
dbutils.widgets.text("catalog", "maven_test")
catalog = dbutils.widgets.get("catalog")

# 1. Define Storage Account and Containers
storage_account = "capstoneadls"
landing_container = "landing"
uc_container = "ucmeta"

# 2. Map tables to your exact folder structure in the landing container
tables_config = {
    "calendar": "operational/calendar/",
    "promotions": "operational/promotions/",
    "regions": "operational/regions/",
    "stores": "operational/stores/",
    "returns": "pos/returns/",
    "transactions": "pos/transactions/"
}

print(f"Starting Multi-Table Bronze Ingestion for Catalog: {catalog}")

# 3. Loop through each entity and ingest using Auto Loader
for table_name, folder_path in tables_config.items():
    # Target Unity Catalog Table
    full_table_name = f"{catalog}.bronze.{table_name}"
    
    # Read from 'landing'
    source_path = f"abfss://{landing_container}@{storage_account}.dfs.core.windows.net/{folder_path}"
    
    # Store Checkpoints & Schema Metadata in 'ucmeta'
    checkpoint_path = f"abfss://{uc_container}@{storage_account}.dfs.core.windows.net/checkpoints/bronze/{table_name}/"
    schema_location = f"abfss://{uc_container}@{storage_account}.dfs.core.windows.net/schema_evolution/bronze/{table_name}/"
    
    print(f"Processing: {full_table_name} from {source_path}")
    
    # Auto Loader Read
    df_raw = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_location)
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
      .option("header", "true")
      .load(source_path)
    )
    
    # Batch Write to Unity Catalog
    query = (df_raw.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .option("mergeSchema", "true") 
      .trigger(availableNow=True) 
      .toTable(full_table_name)
    )
    
    query.awaitTermination()
    print(f"Successfully loaded data into {full_table_name}")

print("All Bronze tables successfully processed.")